***
# Homework 8: HTML and JSON

*Course:** STAT 606 - Computing in Data Science and Statistics SP24

**Name:** Shrivats Sudhir

**NetID:** ssudhir2

**Email:** ssudhir2@wisc.edu

**Collaborators:** Samuel Merten, Amy Merkelz

**Date:** March 31st, 2024
***

In [1]:
from urllib.error import HTTPError
import urllib.request
from bs4 import BeautifulSoup

## 1.) Warmup: Parsing HTML (spent $\approx$ 10 minutes)

**Let’s get started using `BeautifulSoup` with a couple of simple exercises. Both of the following subproblems ask you to retrieve the HTML from a URL given by an argument `s` and determine some simple information about that HTML.** 

**In both functions, you should raise an appropriate error in the event that `s` is not a string, and you should raise an `HTTPError` with an appropriate error message in the event that the success code that
results from trying to access the URL is not code 200.** 

**You may rely on `requests` and/or `urllib` to raise an error for you in the event that `s` is a string but does not encode a valid URL.**

**Write a function `get_page_title` that takes a string `s` as its only argument and returns a string.** 

**Your function should try to treat `s` as a URL, and return the string stored in the title tag of the HTML page stored at that URL.**

**In the unlikely event that the URL has more than one title tag, your function should return the text stored in the first one. If no such title exists, your function should return `None`.**

**A good way to test this function is to simply check that your function returns the string matching the title in your browser– the page title will always be displayed in the tab in which you have the page open.**

In [2]:
def get_page_title(s):

    if not isinstance(s, (str, )):
        raise TypeError(f'URL ({s}) must be a string.')
    
    try:
        response = urllib.request.urlopen(s)
        if response.getcode() != 200:
            raise HTTPError(f'Could not open URL ({s}).')
    except HTTPError as error:
        raise HTTPError(f'HTTP error occurred: {error}')
    
    html = response.read()
    soup = BeautifulSoup(html, 'html.parser')

    return soup.title.string


**Write a function `count_links` that takes a string `s` encoding a URL as its only argument and returns an integer corresponding to the number of hyperlinks on the webpage stored at that URL.** 

**The homework instructions page linked to above is an easy page to test your code on– there are only three links.**

In [3]:
def count_links(s):
    
    if not isinstance(s, (str, )):
        raise TypeError(f'URL ({s}) must be a string.')
    
    try:
        response = urllib.request.urlopen(s)
        if response.getcode() != 200:
            raise HTTPError(f'Could not open URL ({s}).')
    except HTTPError as error:
        raise HTTPError(f'HTTP error occurred: {error}')
    
    html = response.read()
    soup = BeautifulSoup(html, 'html.parser')

    return len(soup.find_all('a'))

## 2.) Retrieving Data from the Web (7 points, spent $\approx$ )

**In this problem, we’ll scrape data from Wikipedia using `BeautifulSoup`. Documentation for `BeauitfulSoup` can be found at *https://www.crummy.com/software/BeautifulSoup/bs4/doc/*.** 

**As mentioned in lecture, there is another package, called requests, which is becoming quite popular, which you are welcome to use for this problem instead, if you wish. Documentation for the requests package can be found at *http://docs.python-requests.org/en/master/*.**

**Suppose you are trying to choose a city to vacation in. A major factor in your decision is weather. Conveniently, lots of weather information is present in the Wikipedia articles for most world cities.** 

**Your job in this problem is to use `BeautifulSoup` to retrieve weather information from Wikipedia articles. We should note that in practice, such information is more easily obtained from, for example, the National Oceanic and Atmospheric Administration (NOAA) in the case of American cities, and from analogous organizations in other countries.**

**Look at a few Wikipedia pages corresponding to cities. For example:**

* *https://en.wikipedia.org/wiki/Madison,_Wisconsin*

* *https://en.wikipedia.org/wiki/Buenos_Aires*

* *https://en.wikipedia.org/wiki/Harbin*

**Note that most city pages include a table titled something like “Climate data for [Cityname] (normals YYYY-YYYY, extremes YYYY-YYYY)” Find a Wikipedia page for a city that includes such a table (such as one of the three above).**

**In your jupyter notebook, open the URL and read the HTML using either `urllib` or `requests`, and parse it with `BeautifulSoup` using the standard parser, `html.parser`.**

**Have a look at the parsed HTML and find the climate data table, which will have the tag table and will contain a child tag `th` containing a string similar to**

`Climate data for [Cityname] (normals YYYY-YYYY, extremes YYYY-YYYY).`

**Find the node in the `BeautifulSoup` object corresponding to this table. What is the structure of this node of the tree (e.g., how many children does the table have, what are their tags, etc.)? You may want to learn a bit about the structure of HTML tables by looking at the resources available on these websites:**

* *https://developer.mozilla.org/en-US/docs/Web/HTML/Element/table*

* *https://www.w3schools.com/html/html_tables.asp*

* *https://www.w3.org/TR/html401/struct/tables.html*

After inspecting element for url = 'https://en.wikipedia.org/wiki/Madison,_Wisconsin', I found the table for `Climate data for {City Name}` and pasted the CSS Selector and XPATH:

CSS Selector: $\texttt{.mw-content-ltr > div:nth-child(110)}$

XPATH: $\texttt{/html/body/div[2]/div/div[3]/main/div[3]/div[3]/div[1]/div[16]}$

I also noticed the following:

* It is enclosed between `<div> <table> <tbody> ... </tbody> </table> </div>`

* The header is enclosed between `<tbody> <tr> ... </tr> </tbody>` and contains `<th colspan="14">`.

* `colspan="14"` always contains the following columns, (1.) Months, (2.) - (13.) Jan - Dec, (14.) Year.

**Write a function `retrieve_climate_table` that takes as its only argument a string representing a URL, and returns the `BeautifulSoup` tag object corresponding to the climate data table (if it exists in the page) and returns `None` if no such table exists on the page.**

**You should check that the URL is retrieved successfully, and raise an error if `urllib2` fails to successfully read the website.** 

**You may notice that some city pages include more than one climate data table or several nested tables (see, for example, https://en.wikipedia.org/wiki/Los_Angeles). In this case, your function may arbitrarily choose one of the tables to return as a BeautifulSoup object.**

In [100]:
def retrieve_climate_table(s):

    if not isinstance(s, (str, )):
        raise TypeError(f'URL ({s}) must be a string.')
    
    try:
        response = urllib.request.urlopen(s)
    except HTTPError as error:
        raise HTTPError(f'HTTP error occurred: {error}')
    
    response = urllib.request.urlopen(s)   
    html = response.read()
    soup = BeautifulSoup(html, 'html.parser')

    for table in soup.find_all('table'):
        th = table.find_all('th')
        for i in th:
            if 'colspan' in i.attrs.keys() and '14' in i.attrs.values():
                return table
    return None    

In [101]:
s = 'https://en.wikipedia.org/wiki/Madison,_Wisconsin'
retrieve_climate_table(s)

<table class="wikitable mw-collapsible" style="width:auto; text-align:center; line-height:1.2em;">
<tbody><tr>
<th colspan="14">Climate data for Madison, Wisconsin (<a href="/wiki/Dane_County_Regional_Airport" title="Dane County Regional Airport">Dane County Regional Airport</a>), 1991–2020 normals,<sup class="reference" id="cite_ref-53"><a href="#cite_note-53">[a]</a></sup> extremes 1869–present<sup class="reference" id="cite_ref-54"><a href="#cite_note-54">[b]</a></sup>
</th></tr>
<tr>
<th scope="row">Month
</th>
<th scope="col">Jan
</th>
<th scope="col">Feb
</th>
<th scope="col">Mar
</th>
<th scope="col">Apr
</th>
<th scope="col">May
</th>
<th scope="col">Jun
</th>
<th scope="col">Jul
</th>
<th scope="col">Aug
</th>
<th scope="col">Sep
</th>
<th scope="col">Oct
</th>
<th scope="col">Nov
</th>
<th scope="col">Dec
</th>
<th scope="col" style="border-left-width:medium">Year
</th></tr>
<tr style="text-align: center;">
<th scope="row" style="height: 16px;">Record high °F (°C)
</th>
<td c

**As you look at some of the climate data tables, you may notice that different cities' tables contain different information. For example, not all cities include snowfall data.** 

**Write a function `list_climate_table_row_names` that takes as its only argument a Wikipedia URL and returns a list of the row names of the climate data table, or returns `None` if no such table exists. The list returned by your function should, ideally, consist solely of Python strings (either Unicode or ASCII), and should not include any BeautifulSoup objects or HTML, and the strings should not have any trailing whitespace (Hint: see the `BeautifulSoup` method get_text()).** 

**The list returned by your script should not include an entry corresponding to the `Climate data for...` row in the table.** 

**Second hint: you are looking for HTML table header (`th`) objects. The HTML attribute `scope` is your friend here, because in the context of an HTML table it tells you when a `th` tag is the header of a row or a column.**

In [122]:
def list_climate_table_row_names(s):
    rows = []
    table_html = retrieve_climate_table(s)
    for row in table_html.find_all('th'):
        if 'scope' in row.attrs.keys() and 'row' in row.attrs.values():
            rows.append(row.text.strip())
    return rows


In [123]:
list_climate_table_row_names(s)

['Month',
 'Record high °F (°C)',
 'Mean maximum °F (°C)',
 'Mean daily maximum °F (°C)',
 'Daily mean °F (°C)',
 'Mean daily minimum °F (°C)',
 'Mean minimum °F (°C)',
 'Record low °F (°C)',
 'Average precipitation inches (mm)',
 'Average snowfall inches (cm)',
 'Average precipitation days (≥ 0.01 in)',
 'Average snowy days (≥ 0.1 in)',
 'Average relative humidity (%)',
 'Mean monthly sunshine hours',
 'Percent possible sunshine']